In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [2]:
# Load dataset
data = pd.read_csv("Churn_Modelling.csv")

# Drop unnecessary columns
data = data.drop(['RowNumber','CustomerId','Surname'], axis=1)

# Label encode Gender
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

# One-hot encode Geography
onehot_encoder_geo = OneHotEncoder()

geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# Combine encoded columns
data = pd.concat([data.drop('Geography', axis=1).reset_index(drop=True),
                  geo_encoded_df.reset_index(drop=True)], axis=1)

# Independent and dependent features
X = data.drop('Exited', axis=1)
y = data['Exited']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [3]:
#define a function to create the model and try different parameters(Keras Classifiers)
def create_model(neurons=32,layers=1):
    model = Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model


In [4]:
#create  a keras classifier
model = KerasClassifier(layers=1,neurons=32,model=create_model,epochs=50,batch_size=10,verbose=1)


In [7]:
#define the grid search parameters
param_grid = {
    'neurons' : [16,32,64,128],
    'layers': [1,2],
    'epochs':[50,100]
}

In [8]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    n_jobs=1,
    cv=3,
    verbose=1
    
)

grid_result = grid.fit(X_train, y_train)

#print the best parameters
print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_))



Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/50
534/534 [==============================] - 1s 2ms/step - loss: 0.4896 - accuracy: 0.7859
Epoch 2/50
534/534 [==============================] - 1s 1ms/step - loss: 0.4287 - accuracy: 0.8112
Epoch 3/50
534/534 [==============================] - 1s 1ms/step - loss: 0.4163 - accuracy: 0.8202
Epoch 4/50
534/534 [==============================] - 1s 1ms/step - loss: 0.4073 - accuracy: 0.8234
Epoch 5/50
534/534 [==============================] - 1s 1ms/step - loss: 0.3999 - accuracy: 0.8279
Epoch 6/50
534/534 [==============================] - 1s 1ms/step - loss: 0.3916 - accuracy: 0.8359
Epoch 7/50
534/534 [==============================] - 1s 1ms/step - loss: 0.3830 - accuracy: 0.8402
Epoch 8/50
534/534 [==============================] - 1s 1ms/step - loss: 0.3742 - accuracy: 0.8429
Epoch 9/50
534/534 [==============================] - 1s 1ms/step - loss: 0.3671 - accuracy: 0.8515
Epoch 10/50
534/534 [==================